# SAML-D case-level — czy przypadek (podgraf kont) rozdziela typologie, których okno konta nie widzi

v1 skończyło się prerejestrowaną gałęzią *„brak wzrostu → sygnatura nie należy do pojedynczego konta → klasyfikacja przypadków”*. Ten notebook testuje tę gałąź, **bez dotykania pipeline'u**.

**Jednostka.** Przypadek = spójna składowa transakcji z `Is_laundering = 1`. Dwie transakcje należą do jednego przypadku, jeśli dzielą konto i dzieli je ≤ 7 dni (łańcuchowo). `Laundering_type` **nie** wyznacza granic przypadku. Służy tylko jako etykieta (typ dominujący; czystość jest raportowana).

**Uczciwe ujęcie zadania.** Granicę przypadku wyznacza flaga podejrzanej transakcji, więc to jest **przypisanie typologii do już wykrytego przypadku** (etap analityka, typologia do SAR), a nie wykrywanie.

**Protokół.**
- Tylko typologie podejrzane, wykonalne w obu reprezentacjach.
- Klasy dzielone na rozłączne pule testowe po 5. W każdej puli porównujemy okna v0 z przypadkami na tych samych zestawach klas.
- Support i query są rozłączne: dla okien po kontach, dla przypadków po grupach kont powiązanych w całym okresie (żeby ten sam pracz nie był po obu stronach).
- Standaryzacja liczona na klasach treningowych puli.
- Metody: `nc_raw` i RF (sondy reprezentacji), ProtoNet MLP (3 seedy × 600 iteracji).

**Kryterium główne (ustalone przed uruchomieniem na SAML-D).** Trafność 2-way (5-shot) na trzech parach strukturalnych, które v0 myli: Bipartite/Stacked Bipartite, Fan_In/Layered_Fan_In, Cycle/Layered_Fan_In. ProtoNet jest trenowany bez obu klas pary. Liczymy średnią po parach, przypadki − okna:
- ProtoNet Δ ≥ 0,10 → **PASS**: freeze `samld_case`, ten sam pipeline.
- ProtoNet Δ < 0,10, ale `nc_raw` albo RF Δ ≥ 0,10 → **INFORMACJA JEST, ProtoNet ją tłumi**.
- wszystko < 0,10 → **kierunek martwy**, zostaje wynik v0.
- mniej niż 2 pary wykonalne w obu reprezentacjach → **nierozstrzygalne**.

**Wynik nagłówkowy (raportowany, porównywalny z v0):** recall 5-way na rozłącznych pulach, dla wszystkich klas i dla sześciu słabych.

**Kontrola mechanizmu:** para BC1/BC2. Predykcja: topologia jej nie rozdziela. Jeśli ProtoNet rośnie tu o ≥ 0,15, zysk pochodzi także z odcięcia normalnej aktywności konta, a nie tylko z topologii. Wtedy piszemy to wprost.

**Kontrola podpisu generatora:** jeśli sam rozmiar (`n_tx`, `n_acc`, czas trwania) daje na parach strukturalnych wynik w granicach 0,03 od pełnych cech, wynik jest podpisem generatora.

**Pary dodatkowe (tylko raportowane):** Scatter-Gather/Layered_Fan_Out i Deposit-Send/Gather-Scatter, czyli najczęstsze pomyłki z explainu v0.

**Rewizja przed uruchomieniem.** Pierwotnie kryterium było recallem w pulach 5-klasowych. Suchy bieg na danych syntetycznych pokazał, że partnerzy mylonej pary mogą trafić do różnych pul i wtedy test nie mierzy rozdzielania pary. Kryterium główne zmieniono na test par, zanim cokolwiek uruchomiono na SAML-D.

Kernel: `fsl (py3.10)`. Czasy do sprawdzenia: wczytanie CSV kilka minut, przypadki około minuty, ewaluacja około 10 minut na CPU.

In [ ]:
import sys, json, time, logging
from pathlib import Path
HOME = Path("/home/jupyter")
sys.path.insert(0, str(HOME / "mlops-meta-learning/fsl-projekt/src"))
from collections import defaultdict
from types import SimpleNamespace
import numpy as np, pandas as pd, torch, networkx as nx
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import RandomForestClassifier
from fsl.config import TrainConfig
from fsl.data.registry import get_loaders
from fsl.data.samld import GroupedEpisodeSampler, _feasible
from fsl.models.mlp import MLP
from fsl.models.protonet import fast_adapt
logging.basicConfig(level=logging.ERROR)

SRC = HOME / "data_cache/saml_d_source/SAML-D.csv"
CASE_CACHE = HOME / "data_cache/samld_case_proto_cases.parquet"
RESULTS = HOME / "data_cache/samld_case_proto_results.json"
LINK_DAYS = 7          # transakcje dzielace konto i odlegle o <= 7 dni -> ten sam przypadek
MIN_PURITY = 0.5       # przypadek bez wiekszosciowej typologii (udzial <= 0.5) odpada

cfg0 = TrainConfig(project="dark-data-discovery", dataset="samld", seed=0, fold=0, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg0)
bundle, sha_v0 = load_frozen(cfg0)
SUS = sorted(bundle["schema"]["suspicious_classes"])
print("v0:", sha_v0[:12], "| okien", len(bundle["windows"]), "| typologii podejrzanych", len(SUS))

## Krok 1 — przypadki (granice bez `Laundering_type`) i ich cechy

In [ ]:
CASE_FEATS = ["n_tx", "n_acc", "duration_days", "n_days", "tx_per_day", "mean_gap",
              "share_src", "share_sink", "share_mid", "max_out_share", "max_in_share", "deg_cv",
              "edge_repeat", "density", "depth", "has_cycle", "reciprocity", "is_bipartite",
              "amt_total", "amt_mean", "amt_cv", "amt_max_med", "r100", "s10k",
              "passthrough", "fwd_1d", "sink_conc", "src_conc", "cash", "xb", "curm"]
SIZE_ONLY = ["n_tx", "n_acc", "duration_days"]


def load_suspicious(src, chunksize=2_000_000):
    # Czyta CSV kawalkami; dzien liczony od minimum CALEGO zbioru (ta sama definicja co freeze v0).
    cols = ["Date", "Sender_account", "Receiver_account", "Amount", "Payment_currency",
            "Received_currency", "Sender_bank_location", "Receiver_bank_location",
            "Payment_type", "Is_laundering", "Laundering_type"]
    dt = {"Sender_account": "int64", "Receiver_account": "int64", "Amount": "float64", "Is_laundering": "int8"}
    dmin, parts, n = None, [], 0
    for ch in pd.read_csv(src, usecols=cols, dtype=dt, chunksize=chunksize):
        n += len(ch)
        d = pd.to_datetime(ch["Date"]).min()
        dmin = d if dmin is None else min(dmin, d)
        parts.append(ch[ch["Is_laundering"] == 1].copy())
    S = pd.concat(parts, ignore_index=True)
    S["day"] = (pd.to_datetime(S["Date"]) - dmin).dt.days.astype("int32")
    return S.reset_index(drop=True), n


def link_components(S, max_gap):
    # Skladowe: transakcje dzielace konto i odlegle o <= max_gap dni (lancuchowo).
    n = len(S)
    t = np.r_[np.arange(n), np.arange(n)]
    acc = np.r_[S["Sender_account"].to_numpy(), S["Receiver_account"].to_numpy()]
    day = np.r_[S["day"].to_numpy(), S["day"].to_numpy()]
    o = np.lexsort((t, day, acc))
    acc, day, t = acc[o], day[o], t[o]
    link = (acc[1:] == acc[:-1]) & ((day[1:] - day[:-1]) <= max_gap)
    A = coo_matrix((np.ones(int(link.sum())), (t[:-1][link], t[1:][link])), shape=(n, n))
    return connected_components(A, directed=False)[1]


def case_features(g):
    s, r = g["Sender_account"].to_numpy(), g["Receiver_account"].to_numpy()
    amt, day = g["Amount"].to_numpy(), g["day"].to_numpy()
    accs = np.unique(np.r_[s, r])
    n_acc, n_tx = len(accs), len(g)
    G = nx.DiGraph()
    G.add_nodes_from(accs.tolist())
    G.add_edges_from(zip(s.tolist(), r.tolist()))
    n_e = G.number_of_edges()
    outd = np.array([G.out_degree(a) for a in accs], float)
    ind = np.array([G.in_degree(a) for a in accs], float)
    src, sink, mid = (ind == 0) & (outd > 0), (outd == 0) & (ind > 0), (ind > 0) & (outd > 0)
    deg = outd + ind
    out_amt = pd.Series(amt).groupby(s).sum()
    in_amt = pd.Series(amt).groupby(r).sum()
    mids = accs[mid]
    if len(mids):
        i_, o_ = in_amt.reindex(mids).to_numpy(), out_amt.reindex(mids).to_numpy()
        passthrough = float(np.mean(np.minimum(i_, o_) / np.maximum(np.maximum(i_, o_), 1e-9)))
        in_days = defaultdict(set)
        for a, d in zip(r, day):
            in_days[a].add(int(d))
        mo = np.isin(s, mids)
        fwd = [(int(d) in in_days[a]) or (int(d) - 1 in in_days[a]) for a, d in zip(s[mo], day[mo])]
        fwd_1d = float(np.mean(fwd)) if fwd else 0.0
    else:
        passthrough, fwd_1d = 0.0, 0.0
    ds = np.sort(day)
    pt = g["Payment_type"].astype(str).str.lower()
    tot = float(amt.sum()) or 1.0
    return {
        "n_tx": n_tx, "n_acc": n_acc,
        "duration_days": int(ds[-1] - ds[0] + 1), "n_days": len(np.unique(ds)),
        "tx_per_day": n_tx / len(np.unique(ds)), "mean_gap": float(np.diff(ds).mean()) if n_tx > 1 else 0.0,
        "share_src": float(src.mean()), "share_sink": float(sink.mean()), "share_mid": float(mid.mean()),
        "max_out_share": float(outd.max() / max(n_acc - 1, 1)), "max_in_share": float(ind.max() / max(n_acc - 1, 1)),
        "deg_cv": float(deg.std() / max(deg.mean(), 1e-9)),
        "edge_repeat": n_tx / max(n_e, 1), "density": n_e / max(n_acc * (n_acc - 1), 1),
        "depth": float(nx.dag_longest_path_length(nx.condensation(G))),
        "has_cycle": float(any(len(c) > 1 for c in nx.strongly_connected_components(G))),
        "reciprocity": float(nx.reciprocity(G)) if n_e else 0.0,
        "is_bipartite": float(nx.is_bipartite(G.to_undirected())),
        "amt_total": tot, "amt_mean": float(amt.mean()),
        "amt_cv": float(amt.std() / max(amt.mean(), 1e-9)),
        "amt_max_med": float(amt.max() / max(np.median(amt), 1e-9)),
        "r100": float((amt % 100 == 0).mean()), "s10k": float(((amt >= 9000) & (amt <= 10000)).mean()),
        "passthrough": passthrough, "fwd_1d": fwd_1d,
        "sink_conc": float(in_amt.max() / tot), "src_conc": float(out_amt.max() / tot),
        "cash": float(pt.str.contains("cash").mean()),
        "xb": float((g["Sender_bank_location"].astype(str) != g["Receiver_bank_location"].astype(str)).mean()),
        "curm": float((g["Payment_currency"].astype(str) != g["Received_currency"].astype(str)).mean()),
    }


if CASE_CACHE.exists():
    C = pd.read_parquet(CASE_CACHE)
    print("przypadki z cache:", CASE_CACHE)
else:
    t0 = time.time()
    S, n_all = load_suspicious(SRC)
    print(f"transakcji: {n_all:,} | podejrzanych: {len(S):,} ({len(S) / n_all:.3%})")
    S["case"] = link_components(S, LINK_DAYS)
    S["grp"] = link_components(S, 10 ** 9)          # konta powiazane w calym okresie -> rozlacznosc support/query
    assert (S.groupby("case")["grp"].nunique() == 1).all(), "przypadek rozpiety na dwie grupy - blad laczenia"
    rows = []
    for cid, g in S.groupby("case", sort=False):
        vc = g["Laundering_type"].astype(str).value_counts()
        f = case_features(g)
        f.update(case=int(cid), grp=int(g["grp"].iloc[0]), label=vc.index[0], purity=float(vc.iloc[0] / len(g)))
        rows.append(f)
    C = pd.DataFrame(rows)
    C.to_parquet(CASE_CACHE, index=False)
    print(f"przypadki policzone w {time.time() - t0:.0f} s -> {CASE_CACHE}")

n0 = len(C)
big = C["n_tx"].max()
print(f"najwiekszy przypadek: {big} transakcji ({big / C['n_tx'].sum():.1%} wszystkich podejrzanych), "
      f"{C.loc[C['n_tx'].idxmax(), 'n_acc']} kont")
C = C[C["purity"] > MIN_PURITY].reset_index(drop=True)
print(f"przypadkow: {n0:,} | po filtrze czystosci > {MIN_PURITY}: {len(C):,}")
stats = (C.groupby("label")
          .agg(cases=("case", "size"), groups=("grp", "nunique"), purity=("purity", "median"),
               n_acc=("n_acc", "median"), n_tx=("n_tx", "median"), depth=("depth", "median"),
               cycle=("has_cycle", "mean"), mid=("share_mid", "median"), days=("duration_days", "median"))
          .round(2))
print(stats.to_string())

## Krok 2 — które typologie da się testować w obu reprezentacjach (STOP, jeśli za mało)

In [ ]:
WEAK = ["Cycle", "Layered_Fan_In", "Bipartite", "Stacked Bipartite",
        "Behavioural_Change_1", "Behavioural_Change_2"]
STRUCT, BEHAV = WEAK[:4], WEAK[4:]
ECFG = SimpleNamespace(n_way=5, k_shot=5, query=5)

Wv = bundle["windows"][bundle["windows"]["label"].isin(SUS)].reset_index(drop=True)
REPS = {
    "win":  {"df": Wv, "cols": bundle["schema"]["feature_names"], "groups": Wv["acc"].to_numpy(),
             "log": set(bundle["schema"]["log1p_features"])},
    "case": {"df": C, "cols": CASE_FEATS, "groups": C["grp"].to_numpy(), "log": None},
    "size": {"df": C, "cols": SIZE_ONLY, "groups": C["grp"].to_numpy(), "log": None},
}


def feasible(rep, classes):
    lab = rep["df"]["label"].to_numpy().astype(str)
    ok = []
    for c in classes:
        _, cnt = np.unique(rep["groups"][lab == c], return_counts=True)
        if _feasible(cnt, ECFG.k_shot, ECFG.query):
            ok.append(c)
    return ok


fw, fc = feasible(REPS["win"], SUS), feasible(REPS["case"], SUS)
ELIG = sorted(set(fw) & set(fc))
for c in SUS:
    print(f"{c:22s} okna {'OK' if c in fw else '--'}  przypadki {'OK' if c in fc else '--'}"
          + ("   *slaba" if c in WEAK else ""))
print(f"\nwykonalne w obu: {len(ELIG)}")
if len(ELIG) < 10:
    raise RuntimeError("STOP: < 10 typologii wykonalnych w obu reprezentacjach - za malo na pule 5 + trening 5")

perm = [str(c) for c in np.random.default_rng(0).permutation(ELIG)]
SPLITS = [perm[i:i + 5] for i in range(0, len(perm), 5)]
if len(SPLITS[-1]) < 5:                 # ostatnia pula dopelniona klasami z poczatku permutacji
    SPLITS[-1] += [c for c in perm if c not in SPLITS[-1]][:5 - len(SPLITS[-1])]
for i, sp in enumerate(SPLITS):
    print(f"pula {i}: {sp}")

## Krok 3 — wynik nagłówkowy: okna v0 vs przypadki na tych samych pulach 5-way

In [ ]:
DEV = torch.device("cpu")


def matrix(rep, train_classes):
    df, cols = rep["df"], rep["cols"]
    X = df[cols].astype("float64").to_numpy().copy()
    heavy = ([j for j, c in enumerate(cols) if c in rep["log"]] if rep["log"] is not None else
             [j for j in range(X.shape[1]) if X[:, j].min() >= 0 and X[:, j].max() > 50])
    X[:, heavy] = np.log1p(X[:, heavy])
    lab = df["label"].to_numpy().astype(str)
    m = np.isin(lab, list(train_classes))            # statystyki tylko z klas treningowych
    mu, sd = X[m].mean(0), X[m].std(0)
    sd[sd == 0] = 1.0
    return ((X - mu) / sd).astype("float32"), lab


def train_protonet(X, lab, groups, classes, seed, iters):
    tr = GroupedEpisodeSampler(X, lab, groups, classes, ECFG, "train", np.random.default_rng(seed))
    torch.manual_seed(seed)
    m = MLP(in_features=X.shape[1], hid=64)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(iters):
        loss, _ = fast_adapt(m, tr.sample(), 5, 5, 5, DEV)
        opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()


def _tally(store, pred, names):
    for p, t in zip(pred, np.repeat(np.arange(5), 5)):
        store[names[t]][0] += int(p == t)
        store[names[t]][1] += 1


def evaluate_pools(n_eval=300, seeds=(0, 1, 2), iters=600, rf_trees=100, log=print):
    res = defaultdict(lambda: defaultdict(lambda: [0, 0]))     # (metoda, reprezentacja) -> klasa -> [trafienia, razem]
    y = np.repeat(np.arange(5), 5)
    for si, test in enumerate(SPLITS):
        train = [c for c in ELIG if c not in test]
        for rn, rep in REPS.items():
            X, lab = matrix(rep, train)
            ts = GroupedEpisodeSampler(X, lab, rep["groups"], test, ECFG, "test",
                                       np.random.default_rng(1000 + si))
            eps = [ts.sample_indexed() for _ in range(n_eval)]
            for data, _, _, names in eps:
                x = data.numpy().reshape(5, 10, -1)
                xs, xq = x[:, :5].reshape(25, -1), x[:, 5:].reshape(25, -1)
                _tally(res[("nc_raw", rn)], ((xq[:, None] - x[:, :5].mean(1)[None]) ** 2).sum(-1).argmin(1), names)
                rf = RandomForestClassifier(n_estimators=rf_trees, random_state=0).fit(xs, y)
                _tally(res[("rf", rn)], rf.predict(xq), names)
            if rn == "size":
                continue                                        # kontrola rozmiaru: tylko sondy, bez ProtoNetu
            for seed in seeds:
                m = train_protonet(X, lab, rep["groups"], train, seed, iters)
                with torch.no_grad():
                    for data, _, _, names in eps:
                        e = m(data).reshape(5, 10, -1)
                        pred = torch.cdist(e[:, 5:].reshape(25, -1), e[:, :5].mean(1)).argmin(1).numpy()
                        _tally(res[("protonet", rn)], pred, names)
        log(f"  pula {si} gotowa: {test}")
    return res


t0 = time.time()
res = evaluate_pools()
print(f"pule: {time.time() - t0:.0f} s\n")

rec = lambda k, c: res[k][c][0] / res[k][c][1] if res[k][c][1] else float("nan")
avg = lambda k, cls: float(np.nanmean([rec(k, c) for c in cls])) if cls else float("nan")
COLS = [("protonet", "win"), ("protonet", "case"), ("nc_raw", "win"), ("nc_raw", "case"),
        ("rf", "win"), ("rf", "case"), ("nc_raw", "size"), ("rf", "size")]
HEAD = " ".join(f"{m[:5]}-{r}".rjust(11) for m, r in COLS)
print(f"{'klasa':22s} {HEAD}")
for c in ELIG:
    print(f"{c:22s} " + " ".join(f"{rec(k, c):11.3f}" for k in COLS) + ("  *" if c in WEAK else ""))
W6 = [c for c in WEAK if c in ELIG]
pools = {f"{m}-{r}": {"weak6": avg((m, r), W6), "all": avg((m, r), ELIG)} for m, r in COLS}
print("\nwynik naglowkowy (recall 5-way na pulach):")
for k, v in pools.items():
    print(f"  {k:14s} wszystkie {v['all']:.3f} | slabe6 {v['weak6']:.3f}")

## Krok 4 — pary mylone w v0 (kryterium główne) i werdykt

In [ ]:
PAIRS = [("Bipartite", "Stacked Bipartite"), ("Fan_In", "Layered_Fan_In"), ("Cycle", "Layered_Fan_In")]
CONTROL = [("Behavioural_Change_1", "Behavioural_Change_2")]
EXTRA = [("Scatter-Gather", "Layered_Fan_Out"), ("Deposit-Send", "Gather-Scatter")]
E2 = SimpleNamespace(n_way=2, k_shot=5, query=5)


def evaluate_pairs(pairs, n_eval=200, seeds=(0, 1, 2), iters=600, rf_trees=100, log=print):
    acc, y2 = {}, np.repeat(np.arange(2), 5)
    for pi, pair in enumerate(pairs):
        if not all(c in ELIG for c in pair):
            log(f"  {pair}: pominieta (klasa niewykonalna w ktorejs reprezentacji)")
            continue
        train = [c for c in ELIG if c not in pair]         # ProtoNet nie widzi zadnej klasy z pary
        for rn, rep in REPS.items():
            X, lab = matrix(rep, train)
            ts = GroupedEpisodeSampler(X, lab, rep["groups"], list(pair), E2, "pair",
                                       np.random.default_rng(2000 + pi))
            eps = [ts.sample_indexed()[0].numpy().reshape(2, 10, -1) for _ in range(n_eval)]
            acc[("nc_raw", rn, pair)] = float(np.mean(
                [((((x[:, 5:].reshape(10, -1)[:, None] - x[:, :5].mean(1)[None]) ** 2).sum(-1)).argmin(1) == y2).mean()
                 for x in eps]))
            acc[("rf", rn, pair)] = float(np.mean(
                [(RandomForestClassifier(n_estimators=rf_trees, random_state=0)
                  .fit(x[:, :5].reshape(10, -1), y2).predict(x[:, 5:].reshape(10, -1)) == y2).mean() for x in eps]))
            if rn == "size":
                continue
            pr = []
            for seed in seeds:
                m = train_protonet(X, lab, rep["groups"], train, seed, iters)
                with torch.no_grad():
                    for x in eps:
                        e = m(torch.from_numpy(x.reshape(20, -1))).reshape(2, 10, -1)
                        pr.append((torch.cdist(e[:, 5:].reshape(10, -1), e[:, :5].mean(1)).argmin(1).numpy() == y2).mean())
            acc[("protonet", rn, pair)] = float(np.mean(pr))
        log(f"  {pair[0]} / {pair[1]} gotowe")
    return acc


t0 = time.time()
pacc = evaluate_pairs(PAIRS + CONTROL + EXTRA)
print(f"pary: {time.time() - t0:.0f} s\n")
print(f"{'para':42s} {HEAD}")
for pair in PAIRS + CONTROL + EXTRA:
    if ("rf", "win", pair) in pacc:
        tag = "  (kontrola)" if pair in CONTROL else ("  (dodatkowa)" if pair in EXTRA else "")
        print(f"{pair[0] + ' / ' + pair[1]:42s} " + " ".join(f"{pacc[(m, r, pair)]:11.3f}" for m, r in COLS) + tag)

P = [p for p in PAIRS if ("protonet", "case", p) in pacc]
mp = lambda m, r, ps: float(np.mean([pacc[(m, r, p)] for p in ps])) if ps else float("nan")
d = {m: mp(m, "case", P) - mp(m, "win", P) for m in ("protonet", "nc_raw", "rf")}
print(f"\nPary strukturalne ({len(P)}), srednia 2-way: ProtoNet okna {mp('protonet', 'win', P):.3f} -> "
      f"przypadki {mp('protonet', 'case', P):.3f}")
print(f"Delta (przypadki - okna): ProtoNet {d['protonet']:+.3f} | nc_raw {d['nc_raw']:+.3f} | RF {d['rf']:+.3f}")

if len(P) < 2:
    verdict = "NIEROZSTRZYGALNE: < 2 pary strukturalne wykonalne w obu reprezentacjach"
elif d["protonet"] >= 0.10:
    verdict = "PASS: freeze samld_case + ten sam pipeline"
elif max(d["nc_raw"], d["rf"]) >= 0.10:
    verdict = "INFORMACJA JEST, ProtoNet ja tlumi"
else:
    verdict = "KIERUNEK MARTWY: zostaje wynik v0"
notes = []
cp = CONTROL[0]
if ("protonet", "case", cp) in pacc:
    d_bc = pacc[("protonet", "case", cp)] - pacc[("protonet", "win", cp)]
    notes.append(f"kontrola BC1/BC2: {d_bc:+.3f} -> " + (
        "zysk (takze) z odciecia normalnej aktywnosci konta, nie tylko z topologii" if d_bc >= 0.15
        else "zgodnie z predykcja, topologia BC nie rozdziela"))
full = max(mp("nc_raw", "case", P), mp("rf", "case", P))
size = max(mp("nc_raw", "size", P), mp("rf", "size", P))
notes.append(f"podpis generatora: sam rozmiar {size:.3f} vs pelne cechy {full:.3f} -> "
             + ("FLAGA: rozdzielenie tlumaczy sam rozmiar" if size >= full - 0.03 else "rozmiar nie wystarcza"))
print("\nWERDYKT:", verdict)
for n in notes:
    print(" -", n)

RESULTS.write_text(json.dumps({
    "v0_sha": sha_v0, "link_days": LINK_DAYS, "min_purity": MIN_PURITY, "n_cases": int(len(C)),
    "eligible": ELIG, "splits": SPLITS, "verdict": verdict, "notes": notes,
    "pairs": {f"{m}-{r}": {f"{p[0]} / {p[1]}": pacc[(m, r, p)] for p in PAIRS + CONTROL + EXTRA
                           if (m, r, p) in pacc} for m, r in COLS},
    "delta_pairs": d, "pools": pools,
    "pools_per_class": {f"{m}-{r}": {c: rec((m, r), c) for c in ELIG} for m, r in COLS},
    "case_stats": json.loads(stats.to_json(orient="index")),
}, indent=2, ensure_ascii=False))
print("wyniki zapisane:", RESULTS)